# SSF3 Recurrent Fragmentation — Walkthrough

End-to-end demonstration of the pipeline on a small, synthetic example dataset
(`data/example/example_compounds.sdf` — two illustrative molecules, **not**
real METLIN records; real METLIN data cannot be redistributed here, see README).

This notebook exercises the same functions used for the manuscript's
corpus-wide analyses, just on a tiny dataset so it runs in seconds.

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts')

from mol_core import split_sdf_records
from recurrent_ion_discovery import match_universal_ion, FAMILIES, N_TOTAL_IONS
from hydrogen_redistribution import classify_compound_corrected
from ion_family_analysis import ions_present_in_compound

print(f'{N_TOTAL_IONS} recurrent ions across {len(FAMILIES)} families loaded.')

## 1. Load the example compounds

In [ ]:
records = list(split_sdf_records('../data/example/example_compounds.sdf'))
print(f'Loaded {len(records)} example compounds.')
for molblock, props in records:
    print(' -', props.get('METLIN ID'))

## 2. Hydrogen-redistribution classification

Using the corrected, dual-mechanism, mass-conserving classifier
(`hydrogen_redistribution.py`). Each peak is classified as `direct`
(consistent with valence-conserving cleavage), `rearranged` (requires net
hydrogen redistribution), `unexplained`, or `precursor`.

In [ ]:
for molblock, props in records:
    result = classify_compound_corrected(molblock, props)
    print(props.get('METLIN ID'), '->', result)

## 3. Recurrent-ion presence

Checking each compound's peaks against the 56-ion vocabulary
(`recurrent_ion_discovery.py`). The toluene example should show the
tropylium cation (C7H7+), the single most prevalent ion in the corpus-wide
analysis (~19% of all METLIN compounds at 40 eV).

In [ ]:
from rearrangement_classifier_legacy import parse_peak_fields

for molblock, props in records:
    peak_groups = parse_peak_fields(props)
    ions = ions_present_in_compound(peak_groups)
    print(props.get('METLIN ID'), '-> recurrent ions detected:', ions)

## 4. Scaling up: full-corpus analysis

The functions above are exactly what `ion_family_analysis.scan_corpus_checkpointed`
runs over the full METLIN 960K corpus (958,450+ compounds), checkpointing
progress to a JSON state file so a multi-hour scan can be safely interrupted
and resumed. See `data/processed/corpus_scan__*.json` for the actual
full-corpus results reported in the manuscript, and
`scripts/ion_family_analysis.py` for the scanning code itself.

Example (not run here — requires the full METLIN 960K SDF corpus, which
is not included in this repository; see README for access):

```python
from ion_family_analysis import scan_corpus_checkpointed
state = scan_corpus_checkpointed(
    sdf_glob='/path/to/metlin_960k_chunks/*.sdf',
    state_path='my_scan_state.json',
    energy_filter='40eV',
)
# Re-run repeatedly (state is checkpointed) until 'ALL FILES COMPLETE' prints.
```

## 5. Reproducing the shared-cation regression finding

`data/processed/shared_cation_regression_FINAL.csv` is the actual,
validated pairs dataset (Family 1 cations only, 20 ppm, METLIN Core v12 --
see `convergence_analysis.py` docstring for why this specific scope, not
all 56 ions, is correct). Loading it directly here rather than
recomputing from raw structures, since that requires the full corpus.

In [ ]:
import pandas as pd
from convergence_analysis import sensitivity_sweep

df = pd.read_csv('../data/processed/shared_cation_regression_FINAL.csv')
print(f'{len(df)} pairs loaded.')

sweep = sensitivity_sweep(df['shared_cations'], df['residual'],
                           cutoffs=[None, 11, 8, 5])
for s in sweep:
    print(f"cutoff={s['cutoff']:>6}  n={s['n']:>5}  "
          f"Pearson r={s['pearson_r']:+.3f} (p={s['pearson_p']:.2g})  "
          f"Spearman r={s['spearman_r']:+.3f} (p={s['spearman_p']:.2g})")

## 6. Regenerating Figure 2

Using the real data loaded above.

In [ ]:
import numpy as np
from convergence_analysis import lowess_bootstrap_ci
from generate_figures import figure2_shared_cation_regression

xgrid = np.arange(0, int(df['shared_cations'].max()) + 1)
central, lower, upper = lowess_bootstrap_ci(df['shared_cations'], df['residual'], xgrid, n_boot=100)

figure2_shared_cation_regression(
    df['shared_cations'], df['residual'],
    {'xgrid': xgrid, 'central': central, 'lower': lower, 'upper': upper},
    sweep,
    '../figures/figure2_regenerated.png',
)
print('Saved ../figures/figure2_regenerated.png')